# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samanashfaq05/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/samanashfaq05/flyrank-ml-internship.git

import os

os.chdir("/content/flyrank-ml-internship")

print("Current directory:", os.getcwd())
print(
    "Dataset exists:",
    os.path.exists("data/raw/content_refresh_anonymized.csv")
)

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 180 (delta 82), reused 101 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 1.87 MiB | 7.81 MiB/s, done.
Resolving deltas: 100% (82/82), done.
Current directory: /content/flyrank-ml-internship
Dataset exists: True


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Random Forest

I chose Random Forest because the content refresh problem has multiple signals that may interact in non-linear ways, such as recent impressions, previous impressions, average position, search volume, and content age. Random Forest can capture these patterns without requiring a simple linear relationship. It also provides feature importance information that can help explain which signals are useful for prioritizing content pages. I will compare its Precision@50 with the Week-4 baseline on the same holdout data.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Rows: 30000
Declining rate: 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: Client-level holdout

I will split the data by client so that pages from the same client do not appear in both the training and test sets. This gives a more honest evaluation because the model must work on clients it did not see during training. I will use the same test data and metric when comparing the model with the baseline.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Features available before the outcome is known
feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

# Keep only columns that actually exist
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols].copy()
y = df["is_declining_label"]
groups = df["client_id"]

# Replace missing numeric values with the median
X = X.fillna(X.median(numeric_only=True))

# Client-level train/test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Clients appearing in both:", len(train_clients & test_clients))
print("Training declining rate:", round(y_train.mean(), 3))
print("Test declining rate:", round(y_test.mean(), 3))
print("Number of features:", len(feature_cols))
print("Features:", feature_cols)

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Clients appearing in both: 0
Training declining rate: 0.55
Test declining rate: 0.511
Number of features: 28
Features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Model: Random Forest

I chose Random Forest because the content refresh problem has several signals that can interact in non-linear ways. The model can learn these patterns without requiring a simple relationship between each feature and decline. I will compare the model with the baseline using the same test data and Precision@50.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

# Train the Random Forest model
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

# Predict probabilities for the declining class
test_scores = model.predict_proba(X_test)[:, 1]

# Precision@K
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

k = 50

model_precision_at_50 = precision_at_k(
    test_scores,
    y_test,
    k
)

base_rate = y_test.mean()

print("Random Forest Precision@50:", round(model_precision_at_50, 3))
print("Test base rate:", round(base_rate, 3))

# Feature importance
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 10 feature importances:")
print(feature_importance.head(10).to_string(index=False))

Random Forest Precision@50: 1.0
Test base rate: 0.511

Top 10 feature importances:
              feature  importance
 impressions_prev_30d    0.208124
 impressions_last_30d    0.172909
      impressions_90d    0.079984
     content_age_days    0.057601
         avg_position    0.056909
days_with_impressions    0.053620
           word_count    0.029571
           char_count    0.029400
    sessions_last_30d    0.027488
      clicks_last_30d    0.024866


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Comparison observation: On the same test set, the Random Forest achieved a Precision@50 of 1.000, while the baseline achieved 0.740. This is an improvement of 0.260 Precision@50 points. The test-set base rate was 0.511. These results show that the model ranked the most likely declining pages more effectively than the transparent baseline on this test split. However, this result is directional and should not be interpreted as proof that the model will always achieve perfect Precision@50 on new data.

In [ ]:
# Create a test-set DataFrame for comparison
test_df = df.iloc[test_idx].copy()

# Model score
test_df["model_score"] = test_scores

# Baseline score
test_df["baseline_score"] = (
    (test_df["impressions_last_30d"] < test_df["impressions_prev_30d"]).astype(int)
    + (test_df["avg_position"] > 20).astype(int)
)

# Calculate Precision@50 for the baseline
baseline_precision_at_50 = precision_at_k(
    test_df["baseline_score"],
    test_df["is_declining_label"],
    50
)

print("Random Forest Precision@50:", round(model_precision_at_50, 3))
print("Baseline Precision@50:", round(baseline_precision_at_50, 3))
print("Test base rate:", round(base_rate, 3))

print("\nComparison:")
print(
    "Model lift over baseline:",
    round(model_precision_at_50 - baseline_precision_at_50, 3)
)

Random Forest Precision@50: 1.0
Baseline Precision@50: 0.74
Test base rate: 0.511

Comparison:
Model lift over baseline: 0.26


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.